In [29]:
from apeGmsh import apeGmsh
from apeGmsh import Numberer, FEMData
import numpy as np
import openseespy.opensees as ops
from pathlib import Path

In [30]:
model_iges_path = Path(r'C:\Users\nmora\Github\pyGmsh\acad') / 'gusset.iges'
assert model_iges_path.exists(), model_iges_path

m1 = apeGmsh(
    model_name='gusset',
    verbose=True
)
m1.initialize()

m1.model.io.load_iges(
    file_path=model_iges_path,
    highest_dim_only=True
)

m1.model.queries.remove_duplicates(tolerance=1)
m1.make_conformal(tolerance=1.0)
m1.model.queries.remove_duplicates(tolerance=1)
m1.model.sync()

m1.physical.add_curve(tags=[6, 7], name="UxUy")
m1.physical.add_curve(tags=[2, 4], name="load")

m1.model.viewer()

(m1.mesh
     .set_size_sources(from_points=False, extend_from_boundary=True)
     .set_global_size(50)
     .generate(dim=2)
     .remove_duplicate_nodes()
     .remove_duplicate_elements())

m1.mesh.viewer()

m1.mesh.partitioning.renumber_mesh(method='rcm')

fem_data = m1.mesh.queries.get_fem_data()

Gmsh version: 4.15.1
[Model] loaded IGES ← gusset.iges  {2: 2}
[Model] remove_duplicates(tolerance=1): merged {0: 4, 1: 3} entities (before={0: 11, 1: 11, 2: 2, 3: 0}, after={0: 7, 1: 8, 2: 2, 3: 0})
[Model] make_conformal(dims=[0, 1, 2], tolerance=1.0): entity delta={} (before={0: 7, 1: 8, 2: 2, 3: 0}, after={0: 7, 1: 8, 2: 2, 3: 0})
[Model] remove_duplicates(tolerance=1): merged {} entities (before={0: 7, 1: 8, 2: 2, 3: 0}, after={0: 7, 1: 8, 2: 2, 3: 0})
[Model] OCC kernel synchronised
[PhysicalGroups] add(dim=1, entities=[6, 7]) → pg_tag=1, name='UxUy'
[PhysicalGroups] add(dim=1, entities=[2, 4]) → pg_tag=2, name='load'
[picker] closed — 0 physical group(s) written, 0 picks in working set
[Mesh] set_size_sources(from_points=False, from_curvature=None, extend_from_boundary=True)
[Mesh] set_global_size(max=50, min=0.0)
[Mesh] generate(dim=2)
remove_duplicate_nodes: no duplicates found (87 nodes unchanged)
[Mesh] remove_duplicate_nodes() removed=0
remove_duplicate_elements: no duplica

In [ ]:
fem_data.

array([[12, 44, 11],
       [17, 45, 16],
       [10, 47,  9],
       [15, 46, 14],
       [44, 46, 11],
       [45, 47, 16],
       [46, 47, 10],
       [15, 47, 46],
       [14, 46, 44],
       [ 9, 47, 45],
       [12, 49, 44],
       [17, 48, 45],
       [11, 46, 10],
       [16, 47, 15],
       [ 9, 45,  8],
       [14, 44, 13],
       [28, 48,  1],
       [ 1, 48, 17],
       [29, 49,  3],
       [ 3, 49, 12],
       [45, 51,  8],
       [44, 50, 13],
       [48, 51, 45],
       [49, 50, 44],
       [ 4, 50, 29],
       [ 8, 51,  2],
       [13, 50,  4],
       [ 2, 51, 28],
       [28, 51, 48],
       [29, 50, 49],
       [71, 83, 32],
       [41, 82, 70],
       [33, 71, 32],
       [41, 70, 40],
       [75, 80, 54],
       [53, 84, 62],
       [62, 78, 53],
       [54, 80, 64],
       [65, 68,  3],
       [65, 77, 68],
       [59, 84, 79],
       [62, 84, 59],
       [63, 64, 57],
       [72, 79, 25],
       [ 3, 68, 67],
       [10, 83, 71],
       [70, 82, 15],
       [59, 7

In [ ]:
import openseespy.opensees as ops

ops.wipe()
ops.model('basic', '-ndm', 2, '-ndf', 3)

ops.timeSeries('Linear', 1)
ops.pattern('Plain', 1, 1)

for node_id, coords in zip(fem_data.node_ids, fem_data.node_coords):
    ops.node(int(node_id), *coords)
    
plate_section = 1
ops.section('ElasticMembranePlateSection', plate_section, 2000000, 0.3, 20, 0)

for 
    ops.element('ShellDKGT', int(elem_id), int(node_i), int(node_j), int(node_k), plate_section)